# 03 — ÉVALUATION FINALE sur test.csv

Responsable : **Jeanne**

> ⚠️ Ce notebook est exécuté **une seule fois**, après le gel du choix du modèle (synchro 3).
> `test.csv` représente les passagers du mois prochain : il n'a servi ni à explorer, ni à choisir les variables, ni à régler le modèle.

Date et heure de l'exécution unique : voir la sortie de la cellule 2

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # pour importer src/ depuis notebooks/

from src.data import load_train, load_test, TARGET, RANDOM_STATE
from src.features import LANDING_FEATURES
from src.economics import gain_net

## 1. Ré-entraînement du modèle retenu sur tout train

In [ ]:
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

from src.features import NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, MODEL_FEATURES, AGE_MINIMUM_SIGNALEMENT


def preparer(df):
    """Mêmes traitements que dans le notebook 02."""
    X = df[MODEL_FEATURES].copy()
    X["Arrival Delay in Minutes"] = X["Arrival Delay in Minutes"].fillna(X["Departure Delay in Minutes"])
    return X


# Le modèle retenu, réentraîné sur la totalité de train
train = load_train()
X_train, y_train = preparer(train), train[TARGET]

modele = make_pipeline(
    ColumnTransformer([
        ("num", "passthrough", NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), MODEL_CATEGORICAL_FEATURES),
    ]),
    HistGradientBoostingClassifier(random_state=RANDOM_STATE),
)
modele.fit(X_train, y_train)

## 2. Application unique à test et performance finale

In [ ]:
# ===== ÉVALUATION FINALE — test.csv lu ici pour la première et unique fois =====
print("Exécution :", datetime.now().strftime("%Y-%m-%d %H:%M"))

test = load_test()
X_test, y_test = preparer(test), test[TARGET]

scores_test = modele.predict_proba(X_test)[:, 1]
mineur_test = (test["Age"] < AGE_MINIMUM_SIGNALEMENT).values   # jamais signalés

# Mêmes règles de comparaison que dans le notebook 02, mineurs exclus partout
rng = np.random.default_rng(RANDOM_STATE)
eco = (X_test["Class"] == "Eco").astype(int)

final = pd.DataFrame([
    gain_net(y_test, np.where(mineur_test, -np.inf, rng.random(len(y_test)))),
    gain_net(y_test, np.where(mineur_test, -np.inf, eco * 10_000 + X_test["Arrival Delay in Minutes"])),
    gain_net(y_test, np.where(mineur_test, -np.inf, scores_test)),
], index=["Hasard", "Règle : Eco puis retard", "Modèle retenu"])

final[["passagers_abordes", "insatisfaits_abordes", "precision", "gain_net_pour_1000"]].round(3)